In [4]:
# ============================
# COMPLETE DMOFF-BASED CAUSAL COUNTERFACTUAL ANALYSIS (WORKING VERSION)
# ============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------- Configuration ----------
CAUSAL_RESULTS_DIR = "results_causal_analysis"
os.makedirs(CAUSAL_RESULTS_DIR, exist_ok=True)

# Define all station paths
STATION_PATHS = {
    "Aljarafe": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Aljarafe Results\results_dmoff_consistent_metrics",
    "Asomadilla": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Asomadilla Results\results_dmoff_consistent_metrics", 
    "Bermejales": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Bermejales Results\results_dmoff_consistent_metrics",
    "RonsadelVAalle": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\RonsadelVAalle Results\results_dmoff_consistent_metrics",
    "Torneo": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Torneo Results\results_dmoff_consistent_metrics"
}

print("=== DMOFF-BASED CAUSAL COUNTERFACTUAL ANALYSIS ===")
print(f"Processing {len(STATION_PATHS)} stations...")

# ---------- Core Causal Analysis Functions ----------

def get_dmoff_features_from_test_data(test_df):
    """Extract features from test data that match DMOFF pattern"""
    # These are the typical features used in DMOFF based on your code
    exclude_cols = ['obs', 'regime', 'pred_dmoff', 'error', 'FECHA_HORA']
    
    # Get all columns that look like features (pollutants, lags, time features)
    feature_cols = []
    for col in test_df.columns:
        if col in exclude_cols or col.startswith('Unnamed'):
            continue
        
        # Include pollutant columns and their lags
        if any(pollutant in col for pollutant in ['DD', 'NO2', 'PM10', 'TMP', 'VV', 'CO', 'O3']):
            feature_cols.append(col)
        
        # Include time features
        elif any(time_feat in col for time_feat in ['hour', 'month', 'weekday', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']):
            feature_cols.append(col)
    
    return feature_cols

def get_model_feature_count(regime_models, regime):
    """Get the number of features expected by the model"""
    if regime in regime_models:
        # Check LightGBM model
        if 'lgb_model' in regime_models[regime]:
            model = regime_models[regime]['lgb_model']
            # For LightGBM, we can check the number of features by trying to predict
            return regime_models[regime]['lgb_scaler'].n_features_in_
        
        # Check Random Forest model
        elif 'rf_model' in regime_models[regime]:
            return regime_models[regime]['rf_scaler'].n_features_in_
    
    return None

def select_compatible_features(available_features, n_expected):
    """Select features that are compatible with the model's expected feature count"""
    if n_expected is None or len(available_features) <= n_expected:
        return available_features
    
    # Priority order for feature selection
    priority_features = []
    
    # 1. Current pollutant measurements (most important for causal analysis)
    current_pollutants = [f for f in available_features if any(p in f for p in ['NO2', 'PM10', 'CO']) and '_lag' not in f]
    priority_features.extend(current_pollutants)
    
    # 2. Lagged pollutants (important for time series)
    lagged_pollutants = [f for f in available_features if any(p in f for p in ['NO2', 'PM10', 'CO', 'O3']) and '_lag' in f]
    priority_features.extend(lagged_pollutants)
    
    # 3. Meteorological features
    meteo_features = [f for f in available_features if any(m in f for m in ['TMP', 'VV', 'DD'])]
    priority_features.extend(meteo_features)
    
    # 4. Time features
    time_features = [f for f in available_features if any(t in f for t in ['hour', 'month', 'weekday'])]
    priority_features.extend(time_features)
    
    # 5. Cyclic time features
    cyclic_features = [f for f in available_features if any(c in f for c in ['sin', 'cos'])]
    priority_features.extend(cyclic_features)
    
    # Remove duplicates and take the first n_expected features
    selected_features = []
    for feature in priority_features:
        if feature not in selected_features and feature in available_features:
            selected_features.append(feature)
        if len(selected_features) >= n_expected:
            break
    
    # If we still need more features, add any remaining ones
    if len(selected_features) < n_expected:
        remaining_features = [f for f in available_features if f not in selected_features]
        selected_features.extend(remaining_features[:n_expected - len(selected_features)])
    
    return selected_features

def describe_regime_characteristics(regime_data):
    """Characterize the meteorological regime for interpretation"""
    characteristics = {}
    
    # Basic statistics for key variables
    key_vars = ['DD', 'NO2', 'PM10', 'TMP', 'VV', 'O3']
    for var in key_vars:
        if var in regime_data.columns:
            characteristics[f'{var}_mean'] = float(regime_data[var].mean())
            characteristics[f'{var}_std'] = float(regime_data[var].std())
    
    # Weather characterization
    if 'TMP' in regime_data.columns:
        avg_temp = regime_data['TMP'].mean()
        if avg_temp > 25:
            characteristics['temp_category'] = 'hot'
        elif avg_temp < 10:
            characteristics['temp_category'] = 'cold'
        else:
            characteristics['temp_category'] = 'moderate'
    
    # Wind characterization
    if 'VV' in regime_data.columns:
        avg_wind = regime_data['VV'].mean()
        if avg_wind < 2:
            characteristics['wind_category'] = 'calm'
        elif avg_wind > 5:
            characteristics['wind_category'] = 'windy'
        else:
            characteristics['wind_category'] = 'moderate'
    
    return characteristics

def dmoff_based_causal_analysis(station_name, test_df, regime_models, target_col='obs'):
    """
    Causal analysis using the trained DMOFF models
    How would emission reductions affect O3 in each regime?
    """
    print(f"\n🔍 Running DMOFF-based causal analysis for {station_name}...")
    
    # Policy scenarios based on real emission control measures
    policy_scenarios = {
        'baseline': {
            'description': 'Current emissions (baseline)',
            'changes': {}  # No changes
        },
        'traffic_reduction_20pct': {
            'description': '20% traffic reduction',
            'changes': {'NO2': -0.20, 'CO': -0.15}  # 20% NO2 reduction, 15% CO reduction
        },
        'industrial_controls_30pct': {
            'description': '30% industrial controls',
            'changes': {'PM10': -0.30, 'NO2': -0.25, 'CO': -0.20}
        },
        'clean_energy_transition': {
            'description': 'Clean energy transition',
            'changes': {'NO2': -0.40, 'PM10': -0.35, 'CO': -0.30}
        },
        'winter_heating_restrictions': {
            'description': 'Winter heating restrictions', 
            'changes': {'PM10': -0.50, 'CO': -0.40}
        }
    }
    
    results = {
        'station': station_name,
        'total_samples': len(test_df),
        'scenarios': {}
    }
    
    # Get all available features from test data
    all_available_features = get_dmoff_features_from_test_data(test_df)
    print(f"   Available features in test data: {len(all_available_features)}")
    print(f"   Sample features: {all_available_features[:10]}...")
    
    for scenario_name, scenario in policy_scenarios.items():
        print(f"  📊 Scenario: {scenario['description']}")
        scenario_results = {'regimes': {}}
        
        # Apply scenario to each regime separately
        regimes_present = sorted(test_df['regime'].unique())
        
        for regime in regimes_present:
            regime_data = test_df[test_df['regime'] == regime].copy()
            
            if len(regime_data) < 5:  # Skip regimes with too few samples
                continue
                
            # Create counterfactual data
            counterfactual_data = regime_data.copy()
            
            # Apply emission reductions to relevant features
            changes_applied = 0
            for pollutant, reduction in scenario['changes'].items():
                # Find all features related to this pollutant (current and lagged)
                pollutant_features = [f for f in all_available_features if pollutant in f and f in counterfactual_data.columns]
                
                for feature in pollutant_features:
                    # Apply reduction: new_value = old_value * (1 + reduction)
                    # reduction is negative (e.g., -0.20 for 20% reduction)
                    counterfactual_data[feature] = counterfactual_data[feature] * (1 + reduction)
                    changes_applied += 1
            
            # For baseline scenario - just record baseline statistics
            if scenario_name == 'baseline':
                scenario_results['regimes'][str(regime)] = {
                    'baseline_mean_o3': float(regime_data[target_col].mean()),
                    'counterfactual_mean_o3': float(regime_data[target_col].mean()),
                    'absolute_change_mean': 0.0,
                    'percent_change_mean': 0.0,
                    'n_samples': int(len(regime_data)),
                    'regime_characteristics': describe_regime_characteristics(regime_data)
                }
                continue
            
            # For policy scenarios: use DMOFF models for counterfactual prediction
            if regime in regime_models and changes_applied > 0:
                # Get the number of features expected by the model
                n_expected_features = get_model_feature_count(regime_models, regime)
                
                if n_expected_features is None:
                    print(f"    ⚠️ Could not determine feature count for regime {regime}, skipping...")
                    continue
                
                # Select compatible features
                compatible_features = select_compatible_features(all_available_features, n_expected_features)
                
                if len(compatible_features) != n_expected_features:
                    print(f"    🔄 Using {len(compatible_features)} features for regime {regime} (model expects {n_expected_features})")
                
                # Try LightGBM model first (primary in DMOFF)
                if 'lgb_model' in regime_models[regime]:
                    try:
                        model_info = regime_models[regime]
                        scaler = model_info['lgb_scaler']
                        model = model_info['lgb_model']
                        
                        # Prepare data with compatible features
                        X_counterfactual = counterfactual_data[compatible_features].values
                        X_baseline = regime_data[compatible_features].values
                        
                        # Transform and predict using DMOFF model
                        X_counterfactual_scaled = scaler.transform(X_counterfactual)
                        counterfactual_predictions = model.predict(X_counterfactual_scaled)
                        
                        X_baseline_scaled = scaler.transform(X_baseline)
                        baseline_predictions = model.predict(X_baseline_scaled)
                        
                        # Calculate impact metrics
                        absolute_change = counterfactual_predictions - baseline_predictions
                        percent_change = (absolute_change / (baseline_predictions + 1e-8)) * 100
                        
                        # Remove extreme outliers for stable statistics
                        valid_mask = (percent_change > -100) & (percent_change < 100)
                        if valid_mask.sum() > 0:
                            percent_change = percent_change[valid_mask]
                            absolute_change = absolute_change[valid_mask]
                            baseline_predictions = baseline_predictions[valid_mask]
                            counterfactual_predictions = counterfactual_predictions[valid_mask]
                        
                        scenario_results['regimes'][str(regime)] = {
                            'baseline_mean_o3': float(np.mean(baseline_predictions)),
                            'counterfactual_mean_o3': float(np.mean(counterfactual_predictions)),
                            'absolute_change_mean': float(np.mean(absolute_change)),
                            'percent_change_mean': float(np.mean(percent_change)),
                            'n_samples': int(len(regime_data)),
                            'n_samples_analyzed': int(len(baseline_predictions)),
                            'regime_characteristics': describe_regime_characteristics(regime_data),
                            'model_used': 'LightGBM',
                            'features_used': len(compatible_features),
                            'expected_features': n_expected_features
                        }
                        
                        change_symbol = "🔻" if np.mean(percent_change) < 0 else "🔺"
                        print(f"    {change_symbol} Regime {regime}: {np.mean(percent_change):+6.1f}% O3 change "
                              f"(baseline: {np.mean(baseline_predictions):5.1f} μg/m³, "
                              f"features: {len(compatible_features)}/{n_expected_features})")
                        
                    except Exception as e:
                        print(f"    ❌ Regime {regime} LightGBM error: {str(e)[:80]}")
                
                # Fallback to Random Forest model if LightGBM fails
                elif 'rf_model' in regime_models[regime]:
                    try:
                        model_info = regime_models[regime]
                        scaler = model_info['rf_scaler']
                        model = model_info['rf_model']
                        
                        # Prepare and predict
                        X_counterfactual = counterfactual_data[compatible_features].values
                        X_baseline = regime_data[compatible_features].values
                        
                        X_counterfactual_scaled = scaler.transform(X_counterfactual)
                        counterfactual_predictions = model.predict(X_counterfactual_scaled)
                        
                        X_baseline_scaled = scaler.transform(X_baseline)
                        baseline_predictions = model.predict(X_baseline_scaled)
                        
                        # Calculate changes
                        absolute_change = counterfactual_predictions - baseline_predictions
                        percent_change = (absolute_change / (baseline_predictions + 1e-8)) * 100
                        
                        valid_mask = (percent_change > -100) & (percent_change < 100)
                        if valid_mask.sum() > 0:
                            percent_change = percent_change[valid_mask]
                            absolute_change = absolute_change[valid_mask]
                            baseline_predictions = baseline_predictions[valid_mask]
                        
                        scenario_results['regimes'][str(regime)] = {
                            'baseline_mean_o3': float(np.mean(baseline_predictions)),
                            'counterfactual_mean_o3': float(np.mean(counterfactual_predictions)),
                            'absolute_change_mean': float(np.mean(absolute_change)),
                            'percent_change_mean': float(np.mean(percent_change)),
                            'n_samples': int(len(regime_data)),
                            'n_samples_analyzed': int(len(baseline_predictions)),
                            'regime_characteristics': describe_regime_characteristics(regime_data),
                            'model_used': 'RandomForest',
                            'features_used': len(compatible_features),
                            'expected_features': n_expected_features
                        }
                        
                        change_symbol = "🔻" if np.mean(percent_change) < 0 else "🔺"
                        print(f"    {change_symbol} Regime {regime}: {np.mean(percent_change):+6.1f}% O3 change "
                              f"(RF model, baseline: {np.mean(baseline_predictions):5.1f} μg/m³)")
                        
                    except Exception as e:
                        print(f"    ❌ Regime {regime} RF error: {str(e)[:80]}")
        
        results['scenarios'][scenario_name] = scenario_results
    
    return results

def create_causal_visualizations(station_name, causal_results, output_dir):
    """Create publication-ready visualizations for causal analysis"""
    print(f"  📈 Creating visualizations for {station_name}...")
    
    # Prepare data for plotting
    plot_data = []
    for scenario_name, scenario_results in causal_results['scenarios'].items():
        for regime, metrics in scenario_results['regimes'].items():
            if scenario_name != 'baseline':  # Skip baseline for change plots
                plot_data.append({
                    'station': station_name,
                    'scenario': scenario_name,
                    'scenario_description': get_scenario_description(scenario_name),
                    'regime': f'Regime {regime}',
                    'percent_change': metrics['percent_change_mean'],
                    'absolute_change': metrics['absolute_change_mean'],
                    'baseline_o3': metrics['baseline_mean_o3'],
                    'counterfactual_o3': metrics['counterfactual_mean_o3'],
                    'n_samples': metrics['n_samples'],
                    'temp_category': metrics['regime_characteristics'].get('temp_category', 'unknown'),
                    'wind_category': metrics['regime_characteristics'].get('wind_category', 'unknown')
                })
    
    if not plot_data:
        print("    ⚠️ No data for visualizations")
        return
    
    plot_df = pd.DataFrame(plot_data)
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    fig.suptitle(f'DMOFF-Based Causal Impact Analysis: {station_name}', fontsize=16, fontweight='bold')
    
    try:
        # Plot 1: Percent change by scenario and regime (Bar plot)
        if not plot_df.empty:
            plt.sca(axes[0, 0])
            sns.barplot(data=plot_df, x='scenario', y='percent_change', hue='regime', palette='viridis')
            plt.title('O₃ Concentration Change by Policy Scenario and Regime', fontweight='bold')
            plt.xticks(rotation=45, ha='right')
            plt.ylabel('O₃ Change (%)')
            plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Plot 2: Policy effectiveness heatmap
        if not plot_df.empty:
            plt.sca(axes[0, 1])
            heatmap_data = plot_df.pivot(index='regime', columns='scenario', values='percent_change')
            sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
                        cbar_kws={'label': 'O₃ Change (%)'}, linewidths=0.5)
            plt.title('Policy Effectiveness Heatmap\n(Negative = Improvement)', fontweight='bold')
        
        # Plot 3: Baseline vs policy O3 levels
        scenario_comparison_data = []
        for scenario_name, scenario_results in causal_results['scenarios'].items():
            for regime, metrics in scenario_results['regimes'].items():
                scenario_comparison_data.extend([
                    {'scenario': get_scenario_description(scenario_name), 
                     'regime': f'Regime {regime}', 
                     'type': 'Baseline', 'o3': metrics['baseline_mean_o3']},
                    {'scenario': get_scenario_description(scenario_name), 
                     'regime': f'Regime {regime}', 
                     'type': 'With Policy', 'o3': metrics['counterfactual_mean_o3']}
                ])
        
        if scenario_comparison_data:
            plt.sca(axes[1, 0])
            scenario_df = pd.DataFrame(scenario_comparison_data)
            # Only show non-baseline scenarios
            scenario_df = scenario_df[scenario_df['scenario'] != 'Current emissions (baseline)']
            if not scenario_df.empty:
                sns.barplot(data=scenario_df, x='scenario', y='o3', hue='type', 
                           palette=['skyblue', 'lightcoral'])
                plt.title('O₃ Levels: Baseline vs Policy Scenarios', fontweight='bold')
                plt.xticks(rotation=45, ha='right')
                plt.ylabel('O₃ Concentration (μg/m³)')
        
        # Plot 4: Regime characteristics
        regime_chars = []
        for scenario_name, scenario_results in causal_results['scenarios'].items():
            for regime, metrics in scenario_results['regimes'].items():
                chars = metrics['regime_characteristics']
                regime_chars.append({
                    'regime': f'Regime {regime}',
                    'avg_temperature': chars.get('TMP_mean', 0),
                    'avg_windspeed': chars.get('VV_mean', 0),
                    'avg_nox': chars.get('NO2_mean', 0),
                    'avg_o3_baseline': metrics['baseline_mean_o3'],
                    'n_samples': metrics['n_samples']
                })
        
        if regime_chars:
            plt.sca(axes[1, 1])
            chars_df = pd.DataFrame(regime_chars).drop_duplicates('regime')
            if not chars_df.empty and len(chars_df) > 1:
                scatter = sns.scatterplot(data=chars_df, x='avg_temperature', y='avg_nox', 
                                        hue='regime', size='avg_o3_baseline', 
                                        sizes=(50, 300), palette='dark', alpha=0.7)
                plt.title('Meteorological Regime Characteristics', fontweight='bold')
                plt.xlabel('Average Temperature (°C)')
                plt.ylabel('Average NO₂ (μg/m³)')
                
                # Add regime labels
                for idx, row in chars_df.iterrows():
                    plt.annotate(row['regime'], (row['avg_temperature'], row['avg_nox']),
                                xytext=(5, 5), textcoords='offset points', fontsize=9)
        
        plt.tight_layout()
        
        # Save figure
        plt.savefig(os.path.join(output_dir, f"causal_analysis_{station_name}.png"), 
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        
        print(f"    ✅ Visualizations saved for {station_name}")
        return fig
        
    except Exception as e:
        print(f"    ❌ Visualization error: {e}")
        plt.close()
        return None

def get_scenario_description(scenario_name):
    """Get human-readable scenario descriptions"""
    descriptions = {
        'baseline': 'Current emissions (baseline)',
        'traffic_reduction_20pct': '20% Traffic Reduction',
        'industrial_controls_30pct': '30% Industrial Controls', 
        'clean_energy_transition': 'Clean Energy Transition',
        'winter_heating_restrictions': 'Winter Heating Restrictions'
    }
    return descriptions.get(scenario_name, scenario_name)

# ---------- Main Processing Function ----------

def process_station_causal_analysis(station_name, station_path):
    """Process causal analysis for a single station using DMOFF models"""
    print(f"\n{'='*60}")
    print(f"CAUSAL ANALYSIS: {station_name}")
    print(f"{'='*60}")
    
    # Create station-specific results directory
    station_causal_dir = os.path.join(CAUSAL_RESULTS_DIR, f"Causal_{station_name}")
    os.makedirs(station_causal_dir, exist_ok=True)
    
    try:
        # Step 1: Load DMOFF artifacts
        print("Step 1: Loading DMOFF artifacts...")
        test_df_path = os.path.join(station_path, "test_predictions_dmoff.csv")
        regime_models_path = os.path.join(station_path, "regime_models.pkl")
        
        if not os.path.exists(test_df_path):
            print(f"❌ Test data not found: {test_df_path}")
            return {'station': station_name, 'error': 'Test data file not found'}
        if not os.path.exists(regime_models_path):
            print(f"❌ Regime models not found: {regime_models_path}")
            return {'station': station_name, 'error': 'Regime models file not found'}
        
        test_df = pd.read_csv(test_df_path)
        regime_models = joblib.load(regime_models_path)
        
        print(f"  ✅ Loaded test data: {len(test_df)} samples")
        print(f"  ✅ Loaded regime models for {len(regime_models)} regimes")
        
        # Step 2: Run DMOFF-based causal analysis
        print("Step 2: Running DMOFF-based causal analysis...")
        causal_results = dmoff_based_causal_analysis(
            station_name, test_df, regime_models, target_col='obs'
        )
        
        # Step 3: Create visualizations
        print("Step 3: Creating visualizations...")
        create_causal_visualizations(station_name, causal_results, station_causal_dir)
        
        # Step 4: Save results
        print("Step 4: Saving results...")
        
        # Convert all keys to strings for JSON serialization
        def make_json_serializable(obj):
            if isinstance(obj, (np.integer, np.int32, np.int64)):
                return int(obj)
            elif isinstance(obj, (np.floating, np.float32, np.float64)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {str(k): make_json_serializable(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [make_json_serializable(item) for item in obj]
            elif isinstance(obj, (np.bool_, bool)):
                return bool(obj)
            else:
                return obj
        
        # Save detailed results
        with open(os.path.join(station_causal_dir, f"causal_results_{station_name}.json"), "w") as f:
            json.dump(make_json_serializable(causal_results), f, indent=2)
        
        # Create a simplified CSV summary for easy analysis
        summary_data = []
        for scenario_name, scenario_results in causal_results['scenarios'].items():
            for regime, metrics in scenario_results['regimes'].items():
                summary_data.append({
                    'station': station_name,
                    'scenario': scenario_name,
                    'scenario_description': get_scenario_description(scenario_name),
                    'regime': regime,
                    'baseline_o3': metrics['baseline_mean_o3'],
                    'counterfactual_o3': metrics['counterfactual_mean_o3'],
                    'absolute_change': metrics['absolute_change_mean'],
                    'percent_change': metrics['percent_change_mean'],
                    'n_samples': metrics['n_samples'],
                    'temp_category': metrics['regime_characteristics'].get('temp_category', 'unknown'),
                    'wind_category': metrics['regime_characteristics'].get('wind_category', 'unknown'),
                    'avg_temperature': metrics['regime_characteristics'].get('TMP_mean', 0),
                    'avg_windspeed': metrics['regime_characteristics'].get('VV_mean', 0),
                    'avg_nox': metrics['regime_characteristics'].get('NO2_mean', 0),
                    'model_used': metrics.get('model_used', 'N/A'),
                    'features_used': metrics.get('features_used', 0)
                })
        
        if summary_data:
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_csv(os.path.join(station_causal_dir, f"causal_summary_{station_name}.csv"), index=False)
            print(f"  ✅ Summary saved with {len(summary_df)} records")
        
        print(f"✅ Causal analysis completed for {station_name}!")
        return causal_results
        
    except Exception as e:
        print(f"❌ Error in causal analysis for {station_name}: {e}")
        import traceback
        traceback.print_exc()
        return {'station': station_name, 'error': str(e)}

# ---------- Main Execution ----------

def main():
    """Main function to run causal analysis for all stations"""
    print("Starting DMOFF-based causal counterfactual analysis...")
    
    all_station_results = {}
    successful_stations = []
    
    # Process each station
    for station_name, station_path in STATION_PATHS.items():
        if os.path.exists(station_path):
            print(f"\n{'='*80}")
            print(f"PROCESSING: {station_name}")
            print(f"{'='*80}")
            
            station_results = process_station_causal_analysis(station_name, station_path)
            all_station_results[station_name] = station_results
            
            if 'error' not in station_results:
                successful_stations.append(station_name)
                print(f"✅ SUCCESS: {station_name}")
            else:
                print(f"❌ FAILED: {station_name}")
        else:
            print(f"❌ Station path not found: {station_path}")
            all_station_results[station_name] = {'error': f"Path {station_path} does not exist"}
    
    # Create comprehensive paper-ready summary
    print(f"\n{'='*80}")
    print("CREATING COMPREHENSIVE PAPER-READY SUMMARY")
    print(f"{'='*80}")
    
    # Compile all results into a master table
    master_data = []
    for station_name, results in all_station_results.items():
        if 'scenarios' in results:
            for scenario_name, scenario_results in results['scenarios'].items():
                for regime, metrics in scenario_results['regimes'].items():
                    if scenario_name != 'baseline':  # Only policy scenarios
                        master_data.append({
                            'Station': station_name,
                            'Scenario': get_scenario_description(scenario_name),
                            'Regime': f'Regime {regime}',
                            'Baseline_O3': f"{metrics['baseline_mean_o3']:.1f}",
                            'Policy_O3': f"{metrics['counterfactual_mean_o3']:.1f}",
                            'Change_μg_m3': f"{metrics['absolute_change_mean']:+.1f}",
                            'Change_Percent': f"{metrics['percent_change_mean']:+.1f}%",
                            'Samples': metrics['n_samples'],
                            'Temperature': f"{metrics['regime_characteristics'].get('TMP_mean', 0):.1f}°C",
                            'Wind_Speed': f"{metrics['regime_characteristics'].get('VV_mean', 0):.1f} m/s",
                            'NO2_Level': f"{metrics['regime_characteristics'].get('NO2_mean', 0):.1f}",
                            'Model_Used': metrics.get('model_used', 'N/A')
                        })
    
    # Create master summary table
    if master_data:
        master_df = pd.DataFrame(master_data)
        master_csv_path = os.path.join(CAUSAL_RESULTS_DIR, "PAPER_READY_CAUSAL_ANALYSIS_TABLE.csv")
        master_df.to_csv(master_csv_path, index=False)
        
        # Print beautiful summary table
        print("\n" + "="*120)
        print("PAPER-READY CAUSAL ANALYSIS RESULTS")
        print("="*120)
        
        display_columns = ['Station', 'Scenario', 'Regime', 'Baseline_O3', 'Policy_O3', 'Change_Percent', 'Samples']
        display_df = master_df[display_columns].copy()
        
        # Format the display
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', None)
        print(display_df.to_string(index=False))
        
        # Calculate overall statistics
        if successful_stations:
            print(f"\n{'='*80}")
            print("OVERALL SUMMARY")
            print(f"{'='*80}")
            print(f"✅ Stations successfully analyzed: {len(successful_stations)}/{len(STATION_PATHS)}")
            print(f"📊 Total policy scenarios analyzed: {len(master_data)}")
            
            # Find most and least effective scenarios
            if not master_df.empty:
                # Convert percentage strings to floats for calculation
                master_df['Change_Percent_Num'] = master_df['Change_Percent'].str.rstrip('%').astype(float)
                scenario_performance = master_df.groupby('Scenario')['Change_Percent_Num'].mean()
                
                if len(scenario_performance) > 0:
                    best_scenario = scenario_performance.idxmin()  # Most negative = best improvement
                    worst_scenario = scenario_performance.idxmax()  # Least negative/positive = worst
                    
                    print(f"🏆 Most effective scenario: {best_scenario} ({scenario_performance[best_scenario]:.1f}% change)")
                    print(f"📉 Least effective scenario: {worst_scenario} ({scenario_performance[worst_scenario]:.1f}% change)")
                    
                    # Calculate average improvement
                    avg_improvement = scenario_performance.mean()
                    print(f"📈 Average O3 change across all scenarios: {avg_improvement:+.1f}%")
    
    print(f"\n📁 All results saved to: {CAUSAL_RESULTS_DIR}")
    print("✅ DMOFF-BASED CAUSAL ANALYSIS COMPLETED SUCCESSFULLY!")
    print("="*100)

# ---------- Run Main Function ----------
if __name__ == "__main__":
    main()

=== DMOFF-BASED CAUSAL COUNTERFACTUAL ANALYSIS ===
Processing 5 stations...
Starting DMOFF-based causal counterfactual analysis...

PROCESSING: Aljarafe

CAUSAL ANALYSIS: Aljarafe
Step 1: Loading DMOFF artifacts...
  ✅ Loaded test data: 8511 samples
  ✅ Loaded regime models for 5 regimes
Step 2: Running DMOFF-based causal analysis...

🔍 Running DMOFF-based causal analysis for Aljarafe...
   Available features in test data: 37
   Sample features: ['DD', 'NO2', 'O3', 'PM10', 'TMP', 'VV', 'hour', 'weekday', 'month', 'hour_sin']...
  📊 Scenario: Current emissions (baseline)
  📊 Scenario: 20% traffic reduction
    🔺 Regime 0:   +0.0% O3 change (baseline:  51.9 μg/m³, features: 36/36)
    🔺 Regime 1:   +0.1% O3 change (baseline:  30.4 μg/m³, features: 36/36)
    🔻 Regime 2:   -0.0% O3 change (baseline:  31.4 μg/m³, features: 36/36)
    🔺 Regime 3:   +0.1% O3 change (baseline:  39.8 μg/m³, features: 36/36)
    🔺 Regime 4:   +0.5% O3 change (baseline:  32.1 μg/m³, features: 36/36)
  📊 Scenario